# CulturaMind — extract & select core documents
Runs `pipeline/extract.py` and `pipeline/analyze.py` on PDFs in your Google Drive. Handles English and Persian.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/lparandl/CulturaMind /content/CulturaMind || git -C /content/CulturaMind pull -q
!pip install -q -r /content/CulturaMind/pipeline/requirements.txt
!apt-get install -y -qq tesseract-ocr tesseract-ocr-fas wamerican-large > /dev/null

In [ ]:
# Edit these paths
PDF_DIR = '/content/drive/MyDrive/Cultural texts '          # folder (and subfolders) with the PDFs
OUT_DIR = '/content/drive/MyDrive/CulturaMind_output'      # extracted JSON + reports go here
PASSAGES = OUT_DIR + '/passages'                           # passage-level analysis
BRIEF   = OUT_DIR + '/brief.txt'                           # project description, English + Persian

In [ ]:
# Step 1: extract text (JSON per PDF; scanned pages are OCR'd; --txt also writes readable copies)
!python /content/CulturaMind/pipeline/extract.py "$PDF_DIR" "$OUT_DIR" --txt --ocr eng+fas

In [ ]:
# Step 2: validate extraction quality (then open validation/review.html and check pages by eye)
!python /content/CulturaMind/pipeline/validate.py "$PDF_DIR" "$OUT_DIR" --wordlist /usr/share/dict/american-english-large
import pandas as pd; pd.read_csv(f'{OUT_DIR}/validation/summary.csv')

In [ ]:
# Step 3: split into ~1,500-word passages (skips front matter, index, catalogues, flagged pages)
!python /content/CulturaMind/pipeline/chunk.py "$OUT_DIR" "$PASSAGES" --words 1500

In [ ]:
# Step 4: NLP analysis and core-set selection
import os
brief = f'--brief "{BRIEF}"' if os.path.exists(BRIEF) else ''
!python /content/CulturaMind/pipeline/analyze.py "$PASSAGES" --select 20 {brief}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{PASSAGES}/report.md', encoding='utf-8').read()))